In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

d    = np.load('/content/oasis1_slices_1ch_all436.npz')
IMGS = d['imgs']
pos  = {sid: i for i, sid in enumerate(d['ids'])}

df = pd.read_excel('/content/oasis_cross-sectional-5708aa0a98d82080.xlsx')
df = df[df['ID'].isin(pos)].copy()
df['img_idx'] = df['ID'].map(pos)
df['subject'] = df['ID'].str.extract(r'(OAS1_\d{4})')[0]
to_label = lambda c: 0 if (pd.isna(c) or c == 0) else (1 if c == 0.5 else 2)
df['label'] = df['CDR'].apply(to_label)

A = df.reset_index(drop=True)                                  # confounded, 436
B = df[df['CDR'].notna()].reset_index(drop=True)               # corrected, 235

cvA = list(StratifiedGroupKFold(5, shuffle=True, random_state=42).split(A, A['label'], groups=A['subject']))
cvB = list(StratifiedKFold(5, shuffle=True, random_state=42).split(B, B['label']))

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([transforms.ToPILImage(), transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=7, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([transforms.ToPILImage(), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class DS(Dataset):
    def __init__(self, img_idx, y, tf): self.i, self.y, self.tf = img_idx, y, tf
    def __len__(self): return len(self.i)
    def __getitem__(self, k):
        img = np.repeat(IMGS[self.i[k]][:, :, None], 3, axis=2)   # grey -> 3 identical channels
        return self.tf(img), torch.tensor(int(self.y[k]))

def train_model(img_idx, y, ncls, seed=42):
    torch.manual_seed(seed)
    dl = DataLoader(DS(img_idx, y, train_tf), batch_size=16, shuffle=True, drop_last=True)
    m = models.resnet18(weights='IMAGENET1K_V1')
    m.fc = nn.Linear(m.fc.in_features, ncls); m = m.to(device)
    cnt = np.bincount(y, minlength=ncls)
    w = torch.tensor(np.sqrt(len(y) / (ncls * cnt)), dtype=torch.float32, device=device)
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)
    def go(ep, o):
        s = torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=ep)
        for _ in range(ep):
            m.train()
            for x, t in dl:
                x, t = x.to(device), t.to(device)
                o.zero_grad(); crit(m(x), t).backward(); o.step()
            s.step()
    for p in m.parameters(): p.requires_grad = False
    for p in m.fc.parameters(): p.requires_grad = True
    go(25, torch.optim.AdamW(m.fc.parameters(), lr=1e-3, weight_decay=1e-4))
    for p in m.parameters(): p.requires_grad = True
    go(15, torch.optim.AdamW([
        {'params': m.layer1.parameters(), 'lr': 1e-6}, {'params': m.layer2.parameters(), 'lr': 1e-6},
        {'params': m.layer3.parameters(), 'lr': 1e-5}, {'params': m.layer4.parameters(), 'lr': 1e-5},
        {'params': m.fc.parameters(), 'lr': 1e-4}], weight_decay=1e-4))
    return m.eval()

def predict(m, img_idx):
    dl = DataLoader(DS(img_idx, np.zeros(len(img_idx), int), eval_tf), batch_size=32)
    with torch.no_grad():
        return np.concatenate([torch.softmax(m(x.to(device)), 1).cpu().numpy() for x, _ in dl])

def oof(C, folds, y, ncls):
    P = np.zeros((len(C), ncls))
    for k, (tr, te) in enumerate(folds, 1):
        m = train_model(C['img_idx'].values[tr], y[tr], ncls)
        P[te] = predict(m, C['img_idx'].values[te])
        print(f'  fold {k} done')
    return P

print(device, IMGS.shape)
print('A:', len(A), np.bincount(A['label']), ' people:', A['subject'].nunique())
print('B:', len(B), np.bincount(B['label']))

In [ ]:
yB  = B['label'].values
yB2 = (yB > 0).astype(int)

print('B 3-class:')
P_B3 = oof(B, cvB, yB, 3)
np.save('/content/oof_B_3class_1ch.npy', P_B3)

print('B binary:')
P_B2 = oof(B, cvB, yB2, 2)
np.save('/content/oof_B_binary_1ch.npy', P_B2)

print('saved', P_B3.shape, P_B2.shape)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, recall_score

def report(name, y, P):
    pred = P.argmax(1)
    auc = roc_auc_score(y, P[:, 1]) if P.shape[1] == 2 else roc_auc_score(y, P, multi_class='ovr', average='macro')
    rec = recall_score(y, pred, average=None)
    print(f'{name:<14} acc {accuracy_score(y, pred):.3f}  macroF1 {f1_score(y, pred, average="macro"):.3f}  '
          f'AUC {auc:.3f}  recall per class {np.round(rec, 3)}')

report('B 3-class', yB,  P_B3)
report('B binary',  yB2, P_B2)

print('\nA 3-class:')
yA   = A['label'].values
P_A3 = oof(A, cvA, yA, 3)
np.save('/content/oof_A_3class_1ch.npy', P_A3)
report('A 3-class', yA, P_A3)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict

auc = lambda y, P: roc_auc_score(y, P[:, 1]) if P.shape[1] == 2 else roc_auc_score(y, P, multi_class='ovr', average='macro')

def lr_oof(X, y, folds):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    return cross_val_predict(clf, X, y, cv=folds, method='predict_proba')

def boot(y, P1, P2, groups=None, iters=2000, seed=0):
    rng = np.random.default_rng(seed)
    units = np.unique(groups) if groups is not None else np.arange(len(y))
    rows = {u: np.where(groups == u)[0] for u in units} if groups is not None else None
    a1, a2, g = [], [], []
    for _ in range(iters):
        pick = rng.choice(units, len(units))
        s = np.concatenate([rows[u] for u in pick]) if groups is not None else pick
        if len(np.unique(y[s])) < len(np.unique(y)): continue
        u1, u2 = auc(y[s], P1[s]), auc(y[s], P2[s])
        a1.append(u1); a2.append(u2); g.append(u2 - u1)
    ci = lambda v: np.percentile(v, [2.5, 97.5])
    return ci(a1), ci(a2), np.mean(g), ci(g)

ageA, ageB = A[['Age']].values, B[['Age']].values
A_age = lr_oof(ageA, yA, cvA)
B_age = lr_oof(ageB, yB, cvB)
B_both = lr_oof(np.column_stack([ageB, np.log(P_B3 + 1e-6)]), yB, cvB)

def row(name, y, Pa, Pm, g=None):
    ca, cm, gm, cg = boot(y, Pa, Pm, g)
    sig = 'SIGNIFICANT' if cg[0] > 0 or cg[1] < 0 else 'not significant'
    print(f'{name:<26} first {auc(y, Pa):.3f} [{ca[0]:.3f},{ca[1]:.3f}]   second {auc(y, Pm):.3f} '
          f'[{cm[0]:.3f},{cm[1]:.3f}]   diff {gm:+.3f} [{cg[0]:+.3f},{cg[1]:+.3f}]  {sig}')

print('Age-only (first) vs MRI (second):')
row('A confounded', yA, A_age, P_A3, A['subject'].values)
row('B corrected',  yB, B_age, P_B3)
print('\nMRI (first) vs age+MRI (second):')
row('B corrected', yB, P_B3, B_both)

mask = A['CDR'].notna().values
print('\nElderly-only rescoring of cohort-A predictions:')
print(f'  MRI  all 436: {auc(yA, P_A3):.3f}   CDR-rated only: {auc(yA[mask], P_A3[mask]):.3f}')
print(f'  age  all 436: {auc(yA, A_age):.3f}   CDR-rated only: {auc(yA[mask], A_age[mask]):.3f}')

In [ ]:
allB = B['img_idx'].values

m3 = train_model(allB, yB, 3)
torch.save({
    'state_dict': m3.state_dict(), 'arch': 'resnet18',
    'classes': ['CN', 'MCI', 'AD'], 'n_classes': 3, 'n_train': len(allB),
    'cv': {'accuracy': 0.562, 'macro_f1': 0.469, 'auc': 0.722, 'auc_ci': [0.670, 0.776], 'folds': 5},
    'cohort': 'OASIS-1, CDR-rated subjects only (n=235: 135 CN, 70 MCI, 30 AD). MCI = CDR 0.5 proxy.',
    'input': 'single mid-axial T1 slice, t88 atlas-registered, skull-stripped, 224x224, grey copied to 3 channels, ImageNet normalisation',
    'note': 'Research prototype. Not a diagnostic tool.',
}, '/content/alz_resnet18_3class_v3.pt')

m2 = train_model(allB, yB2, 2)
torch.save({
    'state_dict': m2.state_dict(), 'arch': 'resnet18',
    'classes': ['CN', 'Impaired'], 'n_classes': 2, 'n_train': len(allB),
    'cv': {'accuracy': 0.660, 'macro_f1': 0.658, 'auc': 0.733, 'folds': 5},
    'cohort': 'OASIS-1, CDR-rated subjects only (n=235: 135 CN, 100 impaired).',
    'input': 'single mid-axial T1 slice, t88 atlas-registered, skull-stripped, 224x224, grey copied to 3 channels, ImageNet normalisation',
    'note': 'Research prototype. Not a diagnostic tool.',
}, '/content/alz_resnet18_binary_v3.pt')

print('saved both')

In [ ]:
import os
from PIL import Image
os.makedirs('/content/samples', exist_ok=True)
for cls, name in enumerate(['cn', 'mci', 'ad']):
    rows = B.index[B['label'] == cls][:2]
    for k, r in enumerate(rows, 1):
        Image.fromarray(IMGS[B.loc[r, 'img_idx']]).save(f'/content/samples/{name}_{k}.png')
!cd /content && rm -f samples.zip && zip -r samples.zip samples
print(sorted(os.listdir('/content/samples')))